In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/7_perceptron.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Rudiments of Machine Learning: Learning a Continuous Function

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 20.05.2026
+ **Authors:** [Dr. Benedikt Zönnchen](https://bzoennchen.github.io/Pages/)

In [ ]:
#@title Setup: install required Python packages

# install packages
%pip install numpy
%pip install scikit-learn
%pip install matplotlib
%pip install seaborn
%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os
    
    folders = ['tests']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

        # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('7_perceptron.ipynb')

## 16 The Perceptron (again)

In this notebook we look at the definition and inner workings of Frank Rosenblatt's perceptron.
It is a little bit differently defined than the version we looked at so for, i.e. the one on [slide 41](https://github.com/aica-wavelab/aica-assignments/blob/main/A1_introduction/ai_ml_introduction_deep_dives.pdf).

It as introduced by Frank Rosenblatt in 1958 and is one of the simplest and most historically significant model in machine learning. It was inspired by a basic question: can a machine learn to classify inputs by adjusting its own parameters, much like a biological neuron strengthens or weakens its synaptic connections?

At its core, a perceptron takes a vector of numerical inputs $x_1, x_2, \ldots x_n$, multiplies each by a *learnable weight* $w_1, w_2, \ldots, w_n$, sums the results, and passes that sum through a step function to produce a binary output — either 0 or 1. If the weighted sum exceeds a certain threshold (or bias), the perceptron "fires" and outputs 1; otherwise it outputs 0. Formally, given an input vector x and a weight vector w, the perceptron computes:

\begin{equation}
  y = \begin{cases}
							1 & \text{if } w_0 + \sum_{k=1}^n  x_k \cdot w_k > 0 \\
							0 & \text{otherwise.}
						\end{cases}
\end{equation}

Sometimes we write this expression as a vector multiplication with $\mathbf{x} = (1, x_1, x_2, \ldots, x_n)^\top$ and $\mathbf{w} = (w_0, w_1, w_2, \ldots, w_n)^\top$:

\begin{equation}
  y = \begin{cases}
							1 & \text{if } \mathbf{x}^\top \mathbf{w} > 0 \\
							0 & \text{otherwise.}
						\end{cases}
\end{equation}

This effectively defines a hyperplane. For $n = 2$ it is defined by all $x_1, x_2$ for which 

$$w_0 + w_1 x_1 + w_2 x_2 = 0$$

holds. Thus, the hyperplane seperates the two classes.

What makes the perceptron remarkable is its learning rule. When the model misclassifies an example, the weights are nudged in the direction that would have produced the correct answer. This update rule is guaranteed to converge to a perfect solution — provided one exists. That last caveat is important: the perceptron can only learn decision boundaries that are linear. It famously fails on problems like XOR, where no single straight line can separate the two classes.


This limitation, highlighted by Minsky and Papert in 1969, dampened enthusiasm for neural network research for years. Yet the perceptron remains a foundational building block. Understanding how it works — and where it breaks down — provides the motivation for multi-layer networks, non-linear activation functions, and ultimately the deep learning architectures used today.

In [ ]:
#@title Preparation 
#@markdown Hidden Code, pls execute

# Imports to get additional functionality
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import seaborn as sns

# Configure the visualization
sns.set_theme(style="whitegrid")
#sns.set_context("talk", font_scale=0.8)
#sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

### 16.1 Generating an artificial problem set

To try out our learning algorithm, we create a simulated learning example:

+ There are 2 classes/categories
+ Each class has two different *features*



---

🗣 **Hint:** As you should know, this is a *classification* task!

---

In [ ]:
n_datapoints = 30 # Number of data points we simulate
separation = 1 # Average distance between the two classes
random_seed = 40 # For the same seed we get the same (pseudo-randomly generated) dataset

In [ ]:
#@title Hidden code to generate and plot random data points, please just execute

# Generate some data
np.random.seed(random_seed)

X = np.random.rand(n_datapoints,2)
y = 1.*(np.random.rand(n_datapoints)<0.5)

X[y==1, 0] += separation
X[y==1, 1] += 0.5*separation
#data -= data.mean(axis=0)

print(f'number of data points:: {len(X)}')
print(f'X i.e. input data (first 10): {X[:5]}')
print(f'y i.e. labels (first 10): {y[:5]}')

def get_plot_lim(X):
    minval = X.min()
    maxval = X.max()
    return [0.95*minval,1.05*maxval]

# Plot the data
def plot_data(X, y, lim, ax):
    
    ax.plot(X[y==0,0], X[y==0,1], 'o', label='Class 0')
    ax.plot(X[y==1,0], X[y==1,1], 'o', label='Class 1')

    ax.set_ylim(lim)
    ax.set_xlim(lim)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    return ax
    
fig, ax = plt.subplots(figsize=(5,5))
_ = plot_data(X, y, get_plot_lim(X), ax)
_ = plt.legend(bbox_to_anchor=(1.05, 1))

### 16.2 Defining your Perceptron

We now define a perceptron that assigns each 2D input point `x` (without the leading 1) to a class (0 or 1) depending on the parameters `w0, w1, w2`.

In [ ]:
def perceptron(x, w0, w1, w2):
    activation = w0 + x[0]*w1 + x[1]*w2
    output = int(activation > 0)
    return output

We randomly initialize our *(learnable) parameters*:

In [ ]:
w0 = -1
w1 = 0
w2 = 1

n_updates =  0 # number of updates / learning steps

### 16.3 Test your untrained Perceptron

Let's test the perceptron for different inputs:

In [ ]:
x1 = [0, 0.5]
x2 = [1, 2.5]

print(f'For the input {x1} the perceptron returns {perceptron(x1, w0, w1, w2)}')
print(f'For the input {x2} the perceptron returns {perceptron(x2, w0, w1, w2)}')

That is, with the perceptron we can classify every point in the feature space.
In 2D the perceptron divides the space into two sides by a boundary.
It draws a distinction.
Let's visualize this!

In [ ]:
#@title Hidden code to generate nice plots, please just execute

import matplotlib

LABEL_CURRENT = 'Current distinction boundary.'
LABEL_OLD = 'Last distinction boundary.'

def plot_boundary(w0, w1, w2, lim, ax, background = True,  **kwargs):
    """ plot the decision boundary paramterized by w within the bounds
        of a square described by lim."""
    
    if (w1==0) and (w2==0):
        x0 = lim
        x1 = [w0, w0] 
    elif w2 == 0:
        # perpendicular
        x0 = [-w0/w1]*2
        x1 = [lim]  
    else:
        x0 = lim
        #x1 = [(x*w0/w1 + theta) for x in x0]
        x1 = [(-x*w1 - w0)/w2 for x in x0]
    
    ax.plot(x0, x1, 'k', **kwargs)
    ax.set_xlim(lim)
    
    weights = True
    if weights:
        norm = np.sqrt(w1**2 + w2**2)
        if norm > 0:
            scale = 1.0  # fixed arrow length
            ax.quiver(0, 0, w1/norm * scale, w2/norm * scale, 
                    angles='xy', scale_units='xy', scale=1, color='green', label='w')

    class_00=perceptron((x0[0], x0[0]), w0, w1, w2),
    if background:
        mi, ma = x0 
        b_x0, b_x1 = x1
        #color1 = 'b' if w0 < w1 else 'r'
        #color2 = 'r' if color1 == 'b' else 'b'
        color1 = 'b' if (class_00 and (b_x0 <0)) else 'r'
        color2 = 'r' if color1 == 'b' else 'b'
        area_0 = matplotlib.patches.Polygon(
            [(mi, b_x0), (ma, b_x1), (ma, ma), (mi, ma), (mi, b_x0)],
            #[(x0[0], x1[0]), (x0[1], x1[1]), 
            #(x0[1], x0[0]), (x0[0], x0[0]), 
            #(x0[0], x1[1]), (x0[0], x0[1]),
            #],
            alpha=0.2, color=color1
        )
        area_1 = matplotlib.patches.Polygon(
            [(mi, b_x0), (ma, b_x1), (ma, mi), (mi, mi), (mi, b_x0)],
            #[(x0[1], x0[1]), (x0[0], x0[1]), (x0[0], x1[0]), 
            #(x0[1], x1[1]), (x0[1], x0[0]), (x0[1], x1[0])],
            alpha=0.2, color=color2
        )
        ax.add_patch(area_0)
        ax.add_patch(area_1)


class DistinctionLinePlotter():

    def __init__(self, data, targets, w0, w1, w2):

        self.data = data
        self.targets = targets
        self.w0_old = w0
        self.w1_old = w1
        self.w2_old = w2


    def plot_update(self, datapoint, w0, w1, w2, lim, old=True, ax=None, legend=True):
        
        if ax == None:
            fig, ax = plt.subplots(figsize=(5,5))
            
        plot_data(self.data, self.targets, lim, ax)
        plot_boundary(w0, w1, w2, lim, ax, label=LABEL_CURRENT)

        if old:
            plot_boundary(
                self.w0_old, self.w1_old, self.w2_old, lim, ax, 
                background = False, alpha=0.2, label=LABEL_OLD
            )
            ax.plot(datapoint[0], datapoint[1], 'or', markersize = 10, mfc='none')
            if legend:
                ax.legend(bbox_to_anchor=(1.05, 1))

        # remember the old values for visualization
        self.w0_old = w0
        self.w1_old = w1
        self.w2_old = w2
        return ax

vis = DistinctionLinePlotter(X, y, w0, w1, w2)
fig, ax = plt.subplots(figsize = (5,5))
lim = get_plot_lim(X)
plot_data(X, y, lim, ax)
_ = plot_boundary(w0, w1, w2, lim, ax, label=LABEL_CURRENT)
_ = plt.legend(bbox_to_anchor=(1.05, 1))

# w0: -0.8, w1: 0.9660855154345203, w2: 0.8480911003723159
# w0 + w1*x + w2*y -0.8

### 16.4 Train your Perceptron

In [ ]:
# Define the hyperparameters
epochs = 20 # Learning steps: number of data points examined for errors and updated if necessary
learning_rate = 0.1 # How strong each update step is
n_updates =  0 # number of updates
n_evaluations =  0 # number of times the perceptron is evaluated

# Initialize your weights randomly
w0 = -1
w1 = 0
w2 = 1

In [ ]:
#@title Learning progress

history = []

for epoch in range(epochs):
    for i in range(len(X)):
        datapoint = X[n_evaluations % len(X)]
        desired_output = y[n_evaluations % len(X)]

        perceptron_output = perceptron(datapoint, w0, w1, w2)
        n_evaluations += 1

        error = desired_output - perceptron_output

        if error != 0:
            w0 = w0 + learning_rate * error
            w1 = w1 + learning_rate * error * datapoint[0]
            w2 = w2 + learning_rate * error * datapoint[1]

            history.append({
                'datapoint': datapoint.copy(),
                'w0': w0, 'w1': w1, 'w2': w2,
                'n_updates': n_updates,
                'n_evaluations': n_evaluations,
                'old': True
            })
            n_updates += 1
            
# final frame without the "old" highlight
history.append({
    'datapoint': datapoint.copy(),
    'w0': w0, 'w1': w1, 'w2': w2,
    'n_updates': n_updates,
    'n_evaluations': n_evaluations,
    'old': False
})

In [ ]:
# Visualize the training phase

plt.ioff()
fig, ax = plt.subplots(figsize = (5,5))
lim = get_plot_lim(X)
_ = plot_boundary(w0, w1, w2, lim, ax, label=LABEL_CURRENT)
#ax.set_legend(bbox_to_anchor=(1.05, 1))

def animate(frame_idx):
    ax.clear()
    frame = history[frame_idx]
    vis.plot_update(frame['datapoint'], frame['w0'], frame['w1'], frame['w2'], lim=lim, old=frame['old'], ax=ax, legend=False)
    ax.set_title(f"#parameter updates {frame['n_updates']}, #perceptron evaluations: {frame['n_evaluations']}")

anim = FuncAnimation(fig, animate, frames=len(history), interval=int(300), repeat=False)
plt.close(fig)
plt.ion()
HTML(anim.to_jshtml())

### 🖍 16.5 Exercises

---

🖍 **Exercise 16.1:**  Vary the learning settings. What do you observe?

  + `learning_rate`: How strongly the weights are adjusted for each classification error
  + `steps`: How many data points are examined for errors
  + Change the initial parameters (`w0`, `w1`, `w2`) of the perceptron
  + Decrease the parameter `separation`

What do you observe?

---

Write down your observations!

---

🖍 **Exercise 16.2:**  Exchange the input data `X` and label for the XOR szenario, that is, 

```python
X = np.array([[0,0], [0,1], [1,0], [1,1]])
```

and 

```python
y = np.array([0, 1, 1, 0])
```

does it still work?

---

As you might be able to observe, the update rule introduced by Frank Rosenblatt in 1958 that we use here is

\begin{align*}
w_0 &= w_0 + \eta \cdot (y - h_{\theta}(x_1, x_2)) \\
w_1 &= w_1 + \eta \cdot (y - h_{\theta}(x_1, x_2)) \cdot x_1 \\
w_2 &= w_2 + \eta \cdot (y - h_{\theta}(x_1, x_2)) \cdot x_2
\end{align*}

where $y \in \{0, 1\}$ is the desired result and $h_{\theta}(x_1,x_2) \in \{0, 1\}$ the actual result of the perceptron.

Therefore, the error $(y - h_{\theta}(x_1, x_2))$ is +1 when the perceptron outputs 0 but should output 1. It is -1 when the perceptron outputs 1 but should output 0 and otherwise 0. Think how the update moves weights around and recognize that the green arrow in the plots is the vector $(w_1, w_2)$!.

---

🖍 **Exercise 16.3:**  Can you give an intuition why the update rule makes sense? 

---

---

🖍 **Exercise 16.4:**  Define a network that uses the `perceptron` function with different weights to solve the XOR problem (do not train it but compute the solution by hand). Use the function `xor_network` below. Remember `perceptron(x, w0, w1, w2)` returns `1` if

$$w_0 + x_0 \cdot w_1 + x_1 \cdot w_2 > 0$$

otherwise `0`.

🗣 **Hints:** You may want to think about weights such that your perceptron realizes an AND, OR and NOT.

---

In [ ]:
def neg(value):
  """This function realizes the NOT. It returns 0 if value is 1 and 0 if value is 1.
  """
  return perceptron((0, value), 0.5, 0, -1)

def xor_network(x):
    # x0 AND not x1
    h1 = ...

    # not x0 AND x0
    h2 = ...

    # OR
    y = perceptron((h1,h2), -0.1, 0.7, 0.7)
    
    return y


# Test
for point in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"{point} => {xor_network(point)}")

In [ ]:
grader.check("q164")

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 16.5:**  What is the difference the linear regression we discussed in [Regression](https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/4_linear_regression.ipynb) and a the use of a perceptron?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

